In [213]:
import pandas as pd
import numpy as np
import requests as rq
import sqlite3 as sql3
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from random import randint
import os

load_dotenv()

NASA_API=os.getenv('NASA_API')
URL = "https://api.nasa.gov/planetary/apod?api_key=" + str(NASA_API)


In [324]:
response = rq.get(URL)

if response.status_code == 200:
    data=response.json()
    print("Request successful")
else:
    print("An error occurred: ", response.status_code)

Request successful


In [98]:
title = data.get('title', '')
date = data.get('date', '01-01-0001')
exp = data.get('explanation', '')
img_url = data.get('url', '')
copyr = data.get('copyright', '')

In [170]:
if conn:
    conn.close()
with sql3.connect('apod_data.db') as conn:
    cursor = conn.cursor()

In [171]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS apod_images (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT UNIQUE NOT NULL,
        date TEXT UNIQUE NOT NULL,
        explanation TEXT NOT NULL,
        url TEXT NOT NULL,
        copyright TEXT
    )
''')

In [305]:
try:
    cursor.execute(f'''
        INSERT OR IGNORE INTO apod_images 
        (title, date, explanation, url, copyright)
        VALUES 
        (?, ?, ?, ?, ?)
        ''', (title, date, exp, img_url, copyr))

    conn.commit()
except Exception as e:
    print("Error: ", e)


In [174]:
cursor.execute('''
    SELECT * FROM apod_images;
''').fetchall()

[(1,
  "Total Lunar Eclipse over Tsé Bit'a'í",
  '2026-03-05',
  "rlier this week, Earth’s shadow swept across the full Moon in the year’s only total lunar eclipse. This stunning sequence combines images showing the Moon’s path across the night sky.  Each lunar image captures our planet’s shadow gradually engulfing the Moon, culminating in its red glow.  Sunlight scatters and refracts as it passes through Earth’s atmosphere toward the Moon. Shorter wavelength light (blue and green) scatters more efficiently, leaving red, orange, and yellow hues to paint the lunar surface. Tsé Bit'a'í (”rock with wings”, also known as Shiprock), located in Navajo Nation, provides a powerful volcanic foreground central to this photo and to stories of Navajo origin, adventure, and heroism. As the first full moon of the lunar new year, this eclipse held significance across cultures. Visible from East Asia to North America, this eclipse united observers across great distances, a cosmic reminder that we shar

In [302]:
conn.close()

In [184]:
def download_img(image_url, date):
    os.makedirs('images/', exist_ok=True)
    
    filename = f"apod_{date}.jpg"
    filepath = os.path.join('images/', filename)
    
    try:
        img_response = rq.get(image_url)
    
        if img_response.status_code == 200:
            with open(filepath, 'wb') as f:
                f.write(img_response.content)
        
            print(f"image saved: {filepath}")
        else:
            print("An error occurred, the image was not saved, \
            status code:", img_response.status_code)
    
    except Exception as e:
        print("Failed to download image")


In [327]:
if ".jpg" in data['url']:
    download_img(data['url'], data['date'])

image saved: images/apod_2026-03-05.jpg


In [328]:
def generate_word():
    max_attempts = 20
    
    for attempt in range(max_attempts):
        
        word_response = rq.get("https://random-words-api.kushcreates.com/api?language=en")
        
        if word_response.status_code != 200:
            continue
            
        try:
            word_data = word_response.json()
            n = randint(0, len(word_data)-1)
            
            random_word = word_data[n]['word']
            
            dict_response = rq.get(f"https://api.dictionaryapi.dev/api/v2/entries/en/{random_word}")
            dict_data = dict_response.json()
            
            if isinstance(dict_data, list):
                print(f"Found valid word: {random_word}")
                return dict_data
            else:
                print(f"Word '{random_word}' not in dictionary, trying again...")
                continue
                
        except Exception as e:
            print("Error:", e)
            continue

    # If the word is default, something went wrong with the word generation
    return "default"

In [329]:
dict_data = generate_word()


Found valid word: griff


In [313]:
dict_data[0]['meanings'][0]['definitions'][0]['synonyms']

[]

In [316]:
word_otd = {
    "word": dict_data[0]['word'],
    "definition": dict_data[0]['meanings'][0]['definitions'][0]['definition'],
    "synonyms": ",".join(dict_data[0]['meanings'][0]['definitions'][0]['synonyms']),
    "antonyms": ",".join(dict_data[0]['meanings'][0]['definitions'][0]['antonyms'])
}

In [303]:
if conn:
    conn.close()
with sql3.connect('apod_data.db') as conn:
    cursor = conn.cursor()

In [304]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS words_dict (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        word TEXT UNIQUE NOT NULL,
        definition TEXT UNIQUE NOT NULL,
        synonyms TEXT NOT NULL,
        antonyms TEXT NOT NULL
    )
''')

In [317]:
try:
    cursor.execute(f'''
        INSERT OR IGNORE INTO words_dict 
        (word, definition, synonyms, antonyms)
        VALUES 
        (?, ?, ?, ?)
        ''', (word_otd['word'], word_otd['definition'], word_otd['synonyms'], word_otd['antonyms']))
    
    conn.commit()
except Exception as e:
    print("Error: ", e)


In [320]:
cursor.execute('''
    SELECT * FROM words_dict;
''').fetchall()

ProgrammingError: Cannot operate on a closed cursor.

In [319]:
cursor.close()